# 09 · Pipeline final (Fases 21-22)

**Objetivo.** Ejecutar el **modelo final** de extremo a extremo con una sola llamada, generar la
submission y verificar dos cosas: (1) que el formato cumple la plantilla oficial y (2) que el
resultado es **reproducible** (mismas predicciones que la submission v06 enviada a Kaggle).

El modelo final es el ensamble de 10 fuentes con meta-modelo enriquecido, documentado en
`DECISIONS.md` (D-010, D-012, D-013, D-014, D-015) y con desempeño CV (leave-one-fold-out) de
**0,5851 ± 0,0042** y **0,60273** en el leaderboard público.

**Garantías metodológicas**: los vectorizadores, los modelos base y el meta-modelo se ajustan
**solo con train**; el test se usa exclusivamente para predecir. Todos los componentes aleatorios
llevan semilla fija.

## 0. Configuración

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config as cfg
from src.config import set_seed

set_seed()
pd.set_option("display.width", 175)
pd.set_option("display.max_columns", 60)
print("raíz del proyecto:", ROOT)

raíz del proyecto: /home/carlos/Git/PLN/KAGGLE_AGE_2026_PUJ


## 1. Ejecución del pipeline

Se llama a `src.pipeline.run()`, que construye las representaciones, ajusta los 10 modelos base,
ajusta el meta-modelo con las matrices out-of-fold y escribe la submission validada.
Tiempo esperado: ~8-10 minutos en CPU (32 hilos).

In [2]:
import time
from src import pipeline

started = time.time()
resumen = pipeline.run(output_path=cfg.SUBMISSIONS / "age_submission_final.csv", verbose=True)
print(f"\ntiempo total: {round(time.time() - started, 1)}s")

representaciones: {'char300+menciones': 317433, 'char700+menciones': 617926, 'char300+menciones+hashtags': 323251, 'word12': 136206} (87.7s)


[base] svm 300k + svm 700k (30.1s)


[base] logreg OvR (154.3s)


/home/carlos/venv/lib/python3.14/site-packages/sklearn/linear_model/_stochastic_gradient.py:733: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


[base] sgd + cnb + hashtags (190.7s)


[base] ordinal + word (222.3s)


[base] mlp densas+svd (469.4s)


{
  "checks": {
    "columns_ok": true,
    "columns_match_sample": true,
    "rows_ok": true,
    "ids_equal_test": true,
    "ids_unique": true,
    "ids_same_order": true,
    "no_nan": true,
    "labels_valid": true
  },
  "distribucion_predicha": {
    "13-17": 17974,
    "18-24": 20845,
    "25-34": 17515,
    "35-49": 16992,
    "50-64": 20060,
    "65-xx": 19506
  },
  "segundos": 582.7
}

tiempo total: 582.7s


## 2. Validación del formato de la submission

In [3]:
print(json.dumps(resumen["checks"], indent=2, ensure_ascii=False))
assert all(resumen["checks"].values()), "alguna comprobación de formato falló"
print("\nTodas las comprobaciones de formato pasan: columnas, orden, filas, ids, etiquetas, NaN.")

{
  "columns_ok": true,
  "columns_match_sample": true,
  "rows_ok": true,
  "ids_equal_test": true,
  "ids_unique": true,
  "ids_same_order": true,
  "no_nan": true,
  "labels_valid": true
}

Todas las comprobaciones de formato pasan: columnas, orden, filas, ids, etiquetas, NaN.


## 3. Verificación de reproducibilidad contra la submission v06

In [4]:
final = pd.read_csv(cfg.SUBMISSIONS / "age_submission_final.csv")
v06 = pd.read_csv(cfg.SUBMISSIONS / "age_submission_v06.csv")
coincidencia = float((final[cfg.TARGET] == v06[cfg.TARGET]).mean())
print(f"coincidencia con la submission v06 (la enviada a Kaggle): {coincidencia:.4%}")
print(f"filas: {len(final):,} | etiquetas distintas: {sorted(final[cfg.TARGET].unique())}")
assert coincidencia == 1.0, "el pipeline no reproduce la submission v06"
print("Reproducibilidad verificada: el pipeline regenera exactamente la submission v06.")

coincidencia con la submission v06 (la enviada a Kaggle): 100.0000%
filas: 112,892 | etiquetas distintas: ['13-17', '18-24', '25-34', '35-49', '50-64', '65-xx']
Reproducibilidad verificada: el pipeline regenera exactamente la submission v06.


## 4. Resumen del modelo final

| Componente | Detalle |
|---|---|
| Representación principal | TF-IDF `char_wb(3,6)` sobre texto **crudo** (sin normalizar) |
| Canales auxiliares | menciones (`@cuenta`), hashtags (`#etiqueta`), palabras (1-2) |
| Modelos base (10) | 2× LinearSVC(C=0.2) · LogReg(OvR) · SGD(log_loss) · ComplementNB · SVM+hashtags · Ridge ordinal · softmax ordinal · LogReg palabra · MLP(densas+SVD) |
| Meta-modelo | StandardScaler + LogisticRegression sobre 90 features (49 scores + 6 del MLP + 35 densas) |
| Validación | StratifiedGroupKFold 5 folds agrupado por clúster de near-duplicados + LOFO |
| CV (LOFO) | **0,5851 ± 0,0042** |
| Leaderboard público | **0,60273** |

**Archivos generados**: `submissions/age_submission_final.csv` (idéntico a la v06),
`experiments/submission_v06_resumen.json` y los artefactos de `reports/`.